# Nexus cycle attribution (v2)

Logic lives in `src/v2/scripts/` — this notebook is the driver.

Constants: `scripts/config.toml`.


In [1]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

# make scripts importable when the kernel cwd is anywhere under the repo
SCRIPTS = Path('/home/ajain/apeksha/pnl_attribution/src/v2/scripts')
ROOT = SCRIPTS.parent  # src/v2
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import CONFIG, get_ch_client
from scripts.day_attribution import (
    attribute_all_books_day,
    attribute_book_day,
    load_library_eod_pnl,
)
from scripts.overnight import compute_outside_cycle_mr

client = get_ch_client()
print('books =', len(CONFIG.book_names))
print('contract size =', CONFIG.contract.size, '| clearing =', CONFIG.contract.clearing_rate)


books = 10
contract size = 1000 | clearing = 0.0019


### Market risk + deal vs mark / all days

Day-total **$** (+ = gain, library m2m sign):
- `transfer_vs_mark` — transfer deal vs **Nexus transfer `mark`** (`≈ −edge`; small).
- `transfer_timing_mr` — `algo.curves` asof(xfer) → asof(cycle start) on received lots. Cycle start = min(intention, earliest transfer), so this is ~0 on the first transfer (lag is in held MR).
- `transfer_pred` / `pca` / `routing` — held MR over the cycle window
- `unexecuted` / `executed_mr` — leftover T / fill wait MR
- `mark_to_exec` — hedge fill vs algo curve at fill (`price/n`)
- `total` — sum of those (intraday cycle slice)
- `overnight` — leftover inventory **outside** cycles (calculated, not a plug):
  close→BOD + BOD→first cycle + last-cycle-end→library EOD
- `library_gross` — `algo.nexus_pnl_attribution` EOD `gross`
- `residual` — `library_gross − total − overnight`

Check: `total + overnight + residual = library_gross`. Residual should shrink if overnight is right.


In [ ]:
BOOK = 'Hedger Spreads'
START = date(2026, 9, 17)
END = date.today()

dates = pd.date_range(START, END, freq='D').date.tolist()
rows = []
for d in dates:
    print(f'=== {d} {BOOK} market risk ===')
    one = attribute_book_day(client, d, BOOK, progress_every=0)
    if one.empty or 'DAY TOTAL' not in one.index:
        print('  (no cycles)')
        continue
    day = one.loc['DAY TOTAL']
    lib = load_library_eod_pnl(client, d, BOOK)
    ovn = compute_outside_cycle_mr(client, d, BOOK)
    rows.append({
        'date': d,
        'transfer_vs_mark': float(day[('$', 'transfer_pred', 'transfer_vs_mark')]),
        'transfer_timing_mr': float(day[('$', 'transfer_pred', 'transfer_timing_mr')]),
        'transfer_pred_mr': float(day[('$', 'transfer_pred', 'held_market_risk')]),
        'pca_mr': float(day[('$', 'pca', 'held_market_risk')]),
        'routing_mr': float(day[('$', 'routing', 'held_market_risk')]),
        'unexecuted_mr': float(day[('$', 'executed', 'held_market_risk')]),
        'executed_mr': float(day[('$', 'executed', 'executed_market_risk')]),
        'mark_to_exec': float(day[('$', 'executed', 'mark_to_exec')]),
        'overnight': ovn['overnight'],
        'library_gross': None if lib is None else lib['gross'],
    })

COMPONENT_COLS = [
    'transfer_vs_mark',
    'transfer_timing_mr',
    'transfer_pred_mr',
    'pca_mr',
    'routing_mr',
    'unexecuted_mr',
    'executed_mr',
    'mark_to_exec',
]

mr_all_days = pd.DataFrame(rows).set_index('date')
mr_all_days['total'] = mr_all_days[COMPONENT_COLS].sum(axis=1)
mr_all_days['residual'] = (
    mr_all_days['library_gross'] - mr_all_days['total'] - mr_all_days['overnight']
)
if len(mr_all_days) > 1:
    mr_all_days.loc['TOTAL'] = mr_all_days.sum(numeric_only=True)

display_cols = COMPONENT_COLS + ['total', 'overnight', 'residual', 'library_gross']
mr_all_days = mr_all_days.reindex(columns=display_cols)
mr_all_days.round(2)


=== 2026-09-17 Hedger Spreads market risk ===
  Hedger Spreads: prefetching market data (476 cycles)…
=== 2026-09-18 Hedger Spreads market risk ===
  Hedger Spreads: prefetching market data (446 cycles)…


### Check positions held + unexecuted = position before

For consecutive cycles on the same session day:

```text
Σ held(transfer_pred, pca, routing) + unexecuted  ==  next cycle position_before
```

`unexecuted = triggered + fills` in `(t_i, t_{i+1}]`. Overnight pairs (last cycle of D → first of D+1) are skipped.

Per day: `lots_missing` = Σ |remaining − next position_before| across cycle pairs.

In [ ]:
from scripts.stages import check_risk_roll_continuity

BOOK = 'Hedger Spreads'
START = date(2026, 9, 17)
END = date.today()
ATOL = 1e-6

dates = pd.date_range(START, END, freq='D').date.tolist()
rows = []
for d in dates:
    print(f'=== {d} {BOOK} held+unexecuted vs next position_before ===')
    one = check_risk_roll_continuity(client, BOOK, d, d, include_overnight=False)
    if one.empty:
        print('  (no cycle pairs)')
        continue
    mismatch = one['abs_diff'] > ATOL
    n_bad = int(mismatch.sum())
    lots = float(one['abs_diff'].sum())
    print(f'  {n_bad} / {len(one)} pairs mismatch, {lots:,.1f} lots missing')
    rows.append({
        'date': d,
        'n_pairs': len(one),
        'n_mismatch': n_bad,
        'lots_missing': lots,
        'max_abs_diff': float(one['abs_diff'].max()),
    })

if not rows:
    roll_by_day = pd.DataFrame(
        columns=['n_pairs', 'n_mismatch', 'lots_missing', 'max_abs_diff']
    )
else:
    roll_by_day = pd.DataFrame(rows).set_index('date')
    if len(roll_by_day) > 1:
        tot = roll_by_day.sum(numeric_only=True)
        tot['max_abs_diff'] = roll_by_day['max_abs_diff'].max()
        tot['n_mismatch'] = int(tot['n_mismatch'])
        tot['n_pairs'] = int(tot['n_pairs'])
        roll_by_day.loc['TOTAL'] = tot

roll_by_day.round({'lots_missing': 2, 'max_abs_diff': 2})